# Project 2, Step 1: Inventory policy inputs from forecast error

**Business question for Project 2:** how much safety stock does each SKU need to hit a target service level, what does that inventory cost, and how much does a better forecast save?

This notebook builds the per-SKU inputs the Excel tool needs. It uses the same store (CA_1), the same 2,717 SKUs, and the same train/test split as Project 1.

**Why forecast error, not demand variability?** Safety stock protects against the part of demand you did not predict. If a forecast tracks demand well, the leftover error is small and you need less buffer. That is the link between Project 1 (forecast accuracy) and Project 2 (inventory cost).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

STORE_ID = "CA_1"
HORIZON = 13          # Project 1 test quarter, kept as a holdout here too
CALIB_WEEKS = 52      # last year of training, used to measure 1-week forecast error
MA_WINDOW = 4         # Project 1 winning method: 4-week moving average

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P1_DIR = PROJECT_DIR.parent / "project-1-forecast-fva" / "data" / "processed"
OUT_DIR = PROJECT_DIR / "data"

weekly = pd.read_parquet(P1_DIR / f"weekly_sku_{STORE_ID}.parquet")
p1_fc = pd.read_parquet(P1_DIR / f"forecasts_test_{STORE_ID}.parquet")
skus = p1_fc["item_id"].unique()   # the 2,717 SKUs scored in Project 1
weekly = weekly[weekly["item_id"].isin(skus)].sort_values(["item_id", "wm_yr_wk"])

weeks = np.sort(weekly["wm_yr_wk"].unique())
test_start = weeks[-HORIZON]
calib_start = weeks[-HORIZON - CALIB_WEEKS]
print(f"SKUs: {len(skus):,}")
print(f"Calibration weeks: {weekly.loc[weekly.wm_yr_wk == calib_start, 'week_start'].iloc[0].date()} "
      f"to {weekly.loc[weekly.wm_yr_wk < test_start, 'week_start'].max().date()} ({CALIB_WEEKS} weeks)")
print(f"Holdout (test) weeks start: {weekly.loc[weekly.wm_yr_wk == test_start, 'week_start'].iloc[0].date()}")

SKUs: 2,717
Calibration weeks: 2015-02-21 to 2016-02-13 (52 weeks)
Holdout (test) weeks start: 2016-02-20


## 1. Measure 1-week-ahead forecast error (rolling backtest)

In a store, the replenishment decision is made every week, so the error that matters is the error of the forecast made **one week before** demand happens. Project 1 used a fixed-origin quarterly forecast (1 to 13 weeks ahead), which mixes horizons, so it is not the right error for safety stock.

For each SKU and each of the last 52 training weeks $t$:
- Naive forecast: $F_t = A_{t-1}$
- Moving average forecast: $F_t = \frac{1}{4}(A_{t-1} + A_{t-2} + A_{t-3} + A_{t-4})$
- Error: $e_t = A_t - F_t$

where $A_t$ = actual units sold in week $t$ and $F_t$ = forecast units for week $t$, made using only weeks before $t$.

Per SKU we keep:
- $\sigma_1 = \sqrt{\frac{1}{n}\sum e_t^2}$: root mean squared error (RMSE) of 1-week forecasts. RMSE is used instead of standard deviation because it also counts bias, and a biased forecast needs more buffer
- $\bar{d}$: average weekly demand over the same 52 weeks
- bias $= \sum(F_t - A_t)/\sum A_t$ (negative = under-forecast)

The holdout quarter is not touched, so it can be used later to test whether the policy hits its target service level.

In [2]:
hist = weekly[weekly["wm_yr_wk"] < test_start].copy()
g = hist.groupby("item_id")["units"]
hist["f_naive"] = g.shift(1)
hist["f_ma"] = g.transform(lambda s: s.shift(1).rolling(MA_WINDOW).mean())

calib = hist[hist["wm_yr_wk"] >= calib_start].copy()
assert calib[["f_naive", "f_ma"]].notna().all().all(), "missing forecasts in calibration window"
assert (calib.groupby("item_id").size() == CALIB_WEEKS).all()

for m in ["naive", "ma"]:
    calib[f"e_{m}"] = calib["units"] - calib[f"f_{m}"]

def sku_stats(d):
    A = d["units"].to_numpy(float)
    out = {
        "avg_weekly_demand": A.mean(),
        "sd_weekly_demand": A.std(ddof=1),
        "zero_week_share": (A == 0).mean(),
    }
    for m in ["naive", "ma"]:
        e = d[f"e_{m}"].to_numpy(float)
        out[f"rmse_{m}"] = np.sqrt(np.mean(e ** 2))
        out[f"bias_{m}"] = -e.sum() / A.sum() if A.sum() > 0 else np.nan
    return pd.Series(out)

stats = calib.groupby("item_id").apply(sku_stats, include_groups=False)
print(f"SKU rows: {len(stats):,}")
stats.describe().T.round(2)

SKU rows: 2,717


,count,mean,std,min,25%,50%,75%,max
avg_weekly_demand,2717.0,10.59,19.64,0.0,2.33,5.29,10.63,333.37
sd_weekly_demand,2717.0,5.79,9.04,0.0,1.97,3.31,5.72,195.94
zero_week_share,2717.0,0.20,0.22,0.0,0.02,0.12,0.33,1.00
rmse_naive,2717.0,5.87,7.47,0.0,2.41,3.86,6.21,166.49
bias_naive,2708.0,0.00,0.07,-1.0,-0.01,-0.00,0.01,2.50
rmse_ma,2717.0,5.38,8.31,0.0,1.98,3.23,5.40,216.48
bias_ma,2708.0,0.01,0.17,-1.0,-0.02,-0.00,0.02,5.69


## 2. Does the better forecast shrink the error that drives safety stock?

Safety stock is proportional to $\sigma_1$, so the ratio of total RMSE (moving average vs naive) is a first estimate of how much safety stock the better forecast could save. The dollar version comes in the Excel tool, once cost and service-level assumptions are set.

In [3]:
meta = (weekly.sort_values("wm_yr_wk").groupby("item_id")
        .agg(dept_id=("dept_id", "first"), cat_id=("cat_id", "first")))
last_price = hist.groupby("item_id")["sell_price"].last().rename("last_train_price")
inputs = meta.join(last_price).join(stats).reset_index()

tot = inputs[["rmse_naive", "rmse_ma"]].sum()
print(f"Sum of 1-week RMSE, naive:      {tot.rmse_naive:,.0f} units")
print(f"Sum of 1-week RMSE, moving avg: {tot.rmse_ma:,.0f} units")
print(f"Moving avg vs naive: {tot.rmse_ma / tot.rmse_naive - 1:+.1%}")
print(f"SKUs where moving avg error is lower: {(inputs.rmse_ma < inputs.rmse_naive).mean():.1%}")

by_cat = inputs.groupby("cat_id")[["rmse_naive", "rmse_ma"]].sum()
by_cat["change"] = by_cat["rmse_ma"] / by_cat["rmse_naive"] - 1
by_cat.style.format({"rmse_naive": "{:,.0f}", "rmse_ma": "{:,.0f}", "change": "{:+.1%}"})

Sum of 1-week RMSE, naive:      15,957 units
Sum of 1-week RMSE, moving avg: 14,625 units
Moving avg vs naive: -8.4%
SKUs where moving avg error is lower: 85.7%


,rmse_naive,rmse_ma,change
cat_id,,,
FOODS,"9,946","9,600",-3.5%
HOBBIES,"2,396","1,957",-18.3%
HOUSEHOLD,"3,615","3,068",-15.1%


## 3. Flag SKUs where the standard formula is weak

The safety stock formula $SS = z \cdot \sigma_L$ assumes forecast error is roughly normal. For slow sellers with many zero weeks, demand is lumpy and the normal assumption breaks down. These SKUs are flagged, not dropped, so the Excel tool can show them separately and the README can be honest about it.

Rule used here: flag if average demand is under 1 unit per week **or** more than 50% of weeks have zero sales.

In [4]:
inputs["slow_mover_flag"] = (inputs["avg_weekly_demand"] < 1) | (inputs["zero_week_share"] > 0.5)
flag = inputs.groupby("cat_id")["slow_mover_flag"].agg(["sum", "mean"])
print(f"Flagged slow movers: {inputs.slow_mover_flag.sum():,} of {len(inputs):,} SKUs "
      f"({inputs.slow_mover_flag.mean():.1%})")
unit_share = (inputs.loc[inputs.slow_mover_flag, "avg_weekly_demand"].sum()
              / inputs["avg_weekly_demand"].sum())
print(f"Share of weekly units from flagged SKUs: {unit_share:.1%}")
flag.rename(columns={"sum": "flagged SKUs", "mean": "share"}).style.format({"share": "{:.1%}"})

Flagged slow movers: 338 of 2,717 SKUs (12.4%)
Share of weekly units from flagged SKUs: 2.4%


,flagged SKUs,share
cat_id,,
FOODS,114,8.9%
HOBBIES,92,18.0%
HOUSEHOLD,132,14.2%


## 4. Export inputs for Excel

Two files:
- `sku_policy_inputs_CA_1.csv`: one row per SKU, the inputs the Excel tool needs
- `holdout_demand_CA_1.csv`: actual weekly demand in the 13-week holdout quarter, to test the policy later

Cost, lead time, service level, holding rate, and ordering cost are **not** set here. They are business assumptions and live on the Excel Assumptions sheet so they can be changed.

In [5]:
cols = ["item_id", "dept_id", "cat_id", "last_train_price", "avg_weekly_demand",
        "sd_weekly_demand", "zero_week_share", "rmse_naive", "rmse_ma",
        "bias_naive", "bias_ma", "slow_mover_flag"]
inputs = inputs[cols].round(4)
inputs.to_csv(OUT_DIR / f"sku_policy_inputs_{STORE_ID}.csv", index=False)

holdout = weekly[weekly["wm_yr_wk"] >= test_start][["item_id", "wm_yr_wk", "week_start", "units"]]
holdout.to_csv(OUT_DIR / f"holdout_demand_{STORE_ID}.csv", index=False)
print(f"Saved {len(inputs):,} SKU rows and {len(holdout):,} holdout SKU-weeks to {OUT_DIR.relative_to(PROJECT_DIR)}/")

Saved 2,717 SKU rows and 35,321 holdout SKU-weeks to data/


## Reading the results
_Fill in after running: How much lower is moving-average error than naive, and in which category? How many SKUs are flagged as slow movers, and what share of units do they carry? What does that mean for how much to trust the safety stock numbers?_

**1. The better forecast leaves less error to buffer against.**
- Over the 52 calibration weeks, the 4-week moving average cut total 1-week forecast error (sum of RMSE) by 8.4% vs naive: 14,625 units vs 15,957 units.
- The improvement is broad, not driven by a few items: the moving average had lower error for 85.7% of SKUs.
- This matches Project 1, where the moving average also beat naive (on the 13-week quarterly forecast). Here it holds at the 1-week horizon that matters for weekly replenishment.

**2. The gain is uneven across categories.**
- HOBBIES: -18.3%, HOUSEHOLD: -15.1%, FOODS: only -3.5%.
- FOODS still matters most in absolute terms, because it carries the largest share of total error (9,946 of 15,957 naive RMSE units). Even a small percentage cut there is a meaningful number of units.
- Since safety stock is proportional to forecast error, I expect the safety stock savings to follow the same pattern. The dollar impact depends on unit prices and is calculated in the Excel model.

**3. How much to trust the safety stock numbers.**
- 338 of 2,717 SKUs (12.4%) are flagged as slow movers (under 1 unit per week, or more than half the weeks with zero sales). They carry only 2.4% of weekly units.
- The safety stock formula assumes roughly normal forecast error. That is reasonable for the other 87.6% of SKUs, which carry 97.6% of units. For slow movers, demand is lumpy and the formula can be wrong in either direction, so their results will be reported separately and treated as rough.
- 9 SKUs sold nothing during the calibration window. Their bias cannot be calculated, and they need a separate stocking rule, not a formula-based one.